# Lesson 30 Lab — From PoC to Canary: The Production Launch Gate

**Puzzle:** What evidence must be true before a successful demo becomes a reversible service release?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A production launch is a state transition backed by artifacts, not a meeting sentiment. Functional output, quality, performance, capacity, observability, security, failure recovery, and rollback must converge on one immutable release identity.


## 0. Predict before running

1. Predict which missing artifact blocks the state machine.
2. Check that each upstream evidence hash is retained.
3. Write the exact canary rollback conditions.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The final notebook reads earlier Chapter 03 artifacts when available, verifies their hashes and required gates, creates a release manifest, and runs a deterministic PoC→load-test→canary→promotion state machine. Missing evidence blocks rather than defaults to pass.

- A release identity includes code, image, model, tokenizer, config, and environment.
- Every stage has explicit evidence and an owner.
- Rollback is tested before promotion, not designed during an incident.


## 2. Derive the mechanism

Each stage consumes evidence and has an exit criterion. PoC establishes native functionality and provenance; load testing establishes the service curve; canary compares SLO/error/quality slices; promotion requires monitoring and rollback readiness. A rollback trigger should be computable from live data, and the previous image/model/config tuple must remain deployable.

### Mechanism at a glance

```mermaid
flowchart LR
  P["PoC: function + provenance"] --> L["load test: service curve"]
  L --> C["canary: live SLO + quality"]
  C --> G{"all immutable gates pass?"}
  G -->|"yes"| R["promote with monitoring"]
  G -->|"no"| B["rollback exact prior release"]
  R --> M["post-launch review"]
  M --> P
```

### Walk it step by step

1. **Bind release identity.** Hash code, image, model, tokenizer, config, and evidence.
2. **Advance by gates.** Require functional, load, security, and recovery proof per stage.
3. **Canary with live comparators.** Evaluate SLO, errors, quality slices, and saturation.
4. **Rollback mechanically.** Keep and rehearse the exact previous deployable tuple.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 30
LESSON_TITLE = 'From PoC to Canary: The Production Launch Gate'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260842
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | a successful demo and manual approval |
| Candidate | evidence-complete PoC, load, canary, promotion, and rollback gates |
| Held constant | current repository artifacts, required lesson set, thresholds, release ID, and no invented passes |
| Measurements | artifact presence/hashes, functional gates, metrics gates, security gates, final stage, blockers, and rollback readiness |
| Evidence | `capacity-model` |

**Experiment:** Aggregate selected executed artifacts into an immutable release gate and state-machine decision.


## 5. Inspect the experiment code

The code reads only canonical JSON artifacts and computes hashes over their bytes. It refuses to infer a pass from prose or from a missing metric.

Do not execute until the code matches the frozen table.


In [2]:
chapter=Path.cwd().parent; required=[6,7,8,10,20,21,25,29]; artifacts={}
for number in required:
    matches=list(chapter.glob(f"{number:02d}-*/artifacts/rtx5090-result.json"))
    if len(matches)==1:
        raw=matches[0].read_bytes(); artifacts[str(number)]={"path":str(matches[0].relative_to(chapter)),
            "sha256":hashlib.sha256(raw).hexdigest(),"payload":json.loads(raw)}
checks={"environment":"6" in artifacts and bool(artifacts["6"]["payload"]["metrics"].get("cli_found")),
 "offline_generation":"7" in artifacts and artifacts["7"]["payload"]["metrics"].get("requests",0)>=1,
 "http_contract":"8" in artifacts and artifacts["8"]["payload"]["metrics"].get("schema_valid") is True,
 "provenance":"10" in artifacts and artifacts["10"]["payload"]["metrics"].get("native_load") is True,
 "benchmark":"20" in artifacts and artifacts["20"]["payload"]["metrics"].get("batches",{}).get("8",{}).get("output_tokens_s",0)>0,
 "observability":"21" in artifacts and artifacts["21"]["payload"]["metrics"].get("metrics_status")==200,
 "diagnostics":"25" in artifacts and artifacts["25"]["payload"]["metrics"].get("first_failing_layer")=="none",
 "security":"29" in artifacts and artifacts["29"]["payload"]["metrics"].get("release_blockers")==0,
 "rollback_rehearsed":False}
stages=[("poc",["environment","offline_generation","provenance"]),("load_test",["benchmark","observability"]),
        ("canary",["http_contract","diagnostics","security"]),("promote",["rollback_rehearsed"])]
final="blocked"; blockers=[]
for stage,names in stages:
    failed=[name for name in names if not checks.get(name,False)]
    if failed: blockers.extend(failed); final=f"blocked_before_{stage}"; break
    final=stage
ready=final=="promote" and all(checks.values()); hashes={key:value["sha256"] for key,value in artifacts.items()}
metrics={"required_artifacts":len(required),"artifacts_present":len(artifacts),
 "artifact_hashes":len(set(hashes.values())),"evidence":{k:{"path":v["path"],"sha256":v["sha256"]} for k,v in artifacts.items()},
 "checks":checks,"gates_passed":sum(checks.values()),"gates_total":len(checks),"final_stage":final,
 "release_ready":ready,"blockers":len(blockers),"blocker_names":blockers,
 "release_id":hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest()[:16]}
analysis=(f"The manifest found {len(artifacts)}/{len(required)} artifacts with {metrics['artifact_hashes']} "
          f"hashes and passed {metrics['gates_passed']}/{metrics['gates_total']} gates. Final stage={final}, "
          f"release_ready={ready}; intentional rollback rehearsal prevents lab-only promotion.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Required artifacts | 8 |
| Artifacts present | 8 |
| Artifact hashes | 8 |
| Gates passed | 8 |
| Gates total | 9 |
| Final stage | blocked_before_promote |
| Release ready | no |
| Blockers | 1 |


## 7. Explain the result

The manifest found 8/8 artifacts with 8 hashes and passed 8/9 gates. Final stage=blocked_before_promote, release_ready=False; intentional rollback rehearsal prevents lab-only promotion.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`capacity-model`**. Measured environment facts feed explicit planning arithmetic. Assumed topology, demand, bandwidth, and reserve fields remain assumptions until a native deployment test.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 30, "title": 'From PoC to Canary: The Production Launch Gate', "environment": ENV,
    "evidence_label": 'capacity-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'A reversible release is the conjunction of independent evidence gates; any missing required artifact correctly leaves the candidate blocked.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 30,
  "title": "From PoC to Canary: The Production Launch Gate",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260842
  },
  "evidence_label": "capacity-model",
  "metrics": {
    "required_artifacts": 8,
    "artifacts_present": 8,
    "artifact_hashes": 8,
    "evidence": {
      "6": {
        "path": "06-installation-compatibility/artifacts/rtx5090-result.json",
        "sha256": "fa8b0c16b597cc8c180782a2e72cc48b6e04c51206d88814d169df071cea66f0"
      },
      "7": {
        "path": "07-offline-llm-api/artifacts/rtx5090-result.json",
        "sha256": "b47a8fe19ef8410483ff12306343fc33a0298ceb5a16a0b6cf06bda0df44ac92"
      },
      "8": {
        "path": "08-openai-compatible-service/artifacts/rtx5090-result.json",
        "sha256": "bb958269675df0db0a604538b45bfce0b

## 9. Make the bounded decision

> A reversible release is the conjunction of independent evidence gates; any missing required artifact correctly leaves the candidate blocked.

**Acceptance/rollback:** Promote only when all required evidence passes and the exact previous release completes a tested rollback within its recovery objective.

**Failure analysis:** Lab artifacts come from one GPU and mostly synthetic traffic. A complete manifest can still be invalid for a different region, topology, model, demand distribution, or compliance scope.


## 10. Extend the evidence

Run the manifest in staging and canary with production routing, inject failures, rehearse rollback, record the review owners, and repeat after any input hash changes.

The full boundary and references are in [`README.md`](README.md).
